### 0.IMPORTS

In [ ]:
from kan import KAN
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import random
import pennylane as qml
from sklearn.preprocessing import StandardScaler, RobustScaler,MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, balanced_accuracy_score, average_precision_score,
    matthews_corrcoef, cohen_kappa_score, brier_score_loss, roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.utils import resample
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer

## 1. SEEDING (REPRODUCIBILITY)

In [2]:
batch_size = 32
random_state = 42
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 2. DATA LOADING & PREPROCESSING

In [ ]:
df = pd.read_csv("IndianLiverPatientDataset(ILPD).csv")
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
df['Sickness'] = df['Sickness'].replace(2, 0)
df.columns

Index(['Age', 'Gender', 'TB', 'DB', 'Alkphos', 'Sgpt', 'Sgot', 'TP', 'ALB',
       'A/G', 'Sickness'],
      dtype='object')

In [ ]:
rows_18 = [
    [18,0,1.8,0.7,178,35,36,6.8,3.6,1.10,1],
    [17,0,0.9,0.2,224,36,45,6.9,4.2,1.55,1],
    [24,0,1.0,0.2,189,52,31,8.0,4.8,1.50,1],
    [60,0,2.2,1.0,271,45,52,6.1,2.9,0.90,0],
    [60,0,0.8,0.2,215,24,17,6.3,3.0,0.90,0],
    [38,1,2.6,1.2,410,59,57,5.6,3.0,0.80,0],
    [35,0,2.0,1.1,226,33,135,6.0,2.7,0.80,0],
    [11,0,0.7,0.1,592,26,29,7.1,4.2,1.40,0],
    [65,0,0.7,0.2,265,30,28,5.2,1.8,0.52,0],
    [36,0,5.3,2.3,145,32,92,5.1,2.6,1.00,0],
    [48,0,0.7,0.2,208,15,30,4.6,2.1,0.80,0],
    [65,0,1.4,0.6,260,28,24,5.2,2.2,0.70,0],
    [62,0,0.6,0.1,160,42,110,4.9,2.6,1.10,0],
    [65,0,0.8,0.2,201,18,22,5.4,2.9,1.10,0],
    [17,1,0.7,0.2,145,18,36,7.2,3.9,1.18,0],
    [62,0,0.7,0.2,162,12,17,8.2,3.2,0.60,0],
    [65,0,1.9,0.8,170,36,43,3.8,1.4,0.58,0],
    [23,1,2.3,0.8,509,28,44,6.9,2.9,0.7,0]  
]
rows_18 = pd.DataFrame(rows_18, columns=df.columns)
rows_18_df = rows_18.copy()
matching_indices = []
for i in range(len(rows_18)):
    mask = (df == rows_18.iloc[i]).all(axis=1)
    idx = df.index[mask].tolist()
    matching_indices.append((i, idx))
matching_indices
drop_idx = [134, 102, 496, 367, 33, 34, 474, 417, 493, 105, 106, 413, 414, 145, 36, 532, 182, 411]
df= df.drop([i for i in drop_idx if i in df.index], errors="ignore")
df=df.drop_duplicates().reset_index(drop=True)

### Train/test split

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["Sickness"],
    random_state=random_state
)

train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.5,                
    stratify=train_df["Sickness"],
    random_state=random_state 
)
X_train_df = train_split_df.drop("Sickness", axis=1)
y_train_df = train_split_df["Sickness"]

X_val_df = val_split_df.drop("Sickness", axis=1)
y_val_df = val_split_df["Sickness"]

X_test_df = test_df.drop("Sickness", axis=1)
y_test_df = test_df["Sickness"]

### Missing value

In [ ]:
imputer = IterativeImputer(random_state=42)
imputer.fit(X_train_df)               

X_train_imp = imputer.transform(X_train_df)
X_val_imp   = imputer.transform(X_val_df)
X_test_imp  = imputer.transform(X_test_df)

### Standard Scaler

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train_imp)

X_train_scaled = scaler.transform(X_train_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_test_imp)

### PCA, FA, LDA

In [ ]:
pca = PCA(n_components=7, random_state=42)
pca.fit(X_train_scaled)

fa = FactorAnalysis(n_components=7, random_state=42)
fa.fit(X_train_scaled)

lda = LinearDiscriminantAnalysis(n_components=1)
lda.fit(X_train_scaled, y_train_df)

X_train_final = np.concatenate([
    pca.transform(X_train_scaled),
    fa.transform(X_train_scaled),
    lda.transform(X_train_scaled)
], axis=1)

X_val_final = np.concatenate([
    pca.transform(X_val_scaled),
    fa.transform(X_val_scaled),
    lda.transform(X_val_scaled)
], axis=1)

X_test_final = np.concatenate([
    pca.transform(X_test_scaled),
    fa.transform(X_test_scaled),
    lda.transform(X_test_scaled)
], axis=1)

###  Convert to tensors + π/2 scaling

In [ ]:
X_train = torch.tensor(X_train_final, dtype=torch.float32)
X_val   = torch.tensor(X_val_final,   dtype=torch.float32)
X_test  = torch.tensor(X_test_final,  dtype=torch.float32)

X_train *= torch.cos(X_train)
X_val   *= torch.cos(X_val)
X_test  *= torch.cos(X_test)

psi = 0.5
X_train *= (psi*np.pi)
X_val   *= (psi*np.pi)
X_test  *= (psi*np.pi)

y_train = torch.tensor(y_train_df.values, dtype=torch.float32).unsqueeze(1)
y_val   = torch.tensor(y_val_df.values,   dtype=torch.float32).unsqueeze(1)
y_test  = torch.tensor(y_test_df.values,  dtype=torch.float32).unsqueeze(1)

###  DataLoaders

In [10]:
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=batch_size,
    shuffle=False
)

print(X_train.size())
print(y_train.size())
print(X_val.size())
print(y_val.size())
print(X_test.size())
print(y_test.size())

torch.Size([221, 15])
torch.Size([221, 1])
torch.Size([222, 15])
torch.Size([222, 1])
torch.Size([111, 15])
torch.Size([111, 1])


## 3. Quantum Layer

In [ ]:
n_qubits = 1
n_qaoa_layers = 13
n_layers = 2

dev = qml.device("default.qubit", wires=n_qubits, seed=seed)

@qml.qnode(dev, interface="torch")
def qnode(inputs,qaoa_weights, weights):
    qml.QAOAEmbedding(inputs, qaoa_weights, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {
    "qaoa_weights": (n_qaoa_layers, 2*n_qubits-1),
    "weights": (n_layers, n_qubits),
}

## 4. BINARY FOCAL LOSS

In [16]:
import torch
import torch.nn as nn

class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.3, gamma=1.0, reduction="mean", eps=1e-7):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.eps = eps

    def forward(self, y_pred, y_true):
        y_true = y_true.float()
        y_pred = torch.clamp(y_pred, self.eps, 1.0 - self.eps)

        pos_loss = - self.alpha * (1 - y_pred) ** self.gamma * y_true * torch.log(y_pred)
        neg_loss = - (1 - self.alpha) * y_pred ** self.gamma * (1 - y_true) * torch.log(1 - y_pred)

        loss = pos_loss + neg_loss

        if self.reduction == "mean":
            return loss.mean()
        return loss.sum()


## 5. MODEL ARCHITECTURE

In [ ]:
class HybridKANQuantumModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.kan = KAN(
            width=[input_dim ,16,8, n_qubits],  
            grid=7,      
            k=3,         
            seed=seed
        )
        self.q_layer = qml.qnn.TorchLayer(qnode, weight_shapes)
        self.fc_out = nn.Linear(n_qubits, 1)
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    def forward(self, x):
        x = self.kan(x)      
        x = self.q_layer(x)   
        x = self.fc_out(x)
        return torch.sigmoid(x)

## 6. CLASS WEIGHTS SMOOTHING

In [ ]:
def effective_num_weight(n, beta=0.99):
    return (1 - beta) / (1 - beta ** n)

y_train_np = y_train.numpy().flatten()
n0 = (y_train_np == 0).sum()
n1 = (y_train_np == 1).sum()

# ENS origin
w0 = effective_num_weight(n0, beta=0.99)
w1 = effective_num_weight(n1, beta=0.99)

# Power scaling 
gamma = 4.5
w0 = w0 ** gamma
w1 = w1 ** gamma

mean_w = (w0 + w1) / 2
class_weight_dict = {
    0: float(w0 / mean_w),
    1: float(w1 / mean_w)
}

print("ENS + smooth weights:", class_weight_dict)

ENS + smooth weights: {0: 1.8578762822415118, 1: 0.14212371775848828}


## 7. TRAINING SETUP

In [ ]:
model = HybridKANQuantumModel(X_train.shape[1])

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = BinaryFocalLoss(alpha=0.3, gamma= 1.0)

patience = 1000
best_loss = np.inf
counter = 0
best_state = None


checkpoint directory created: ./model
saving model version 0.0


## 8. TRAINING LOOP

In [ ]:
epochs = 200
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for x, y in train_loader:
        optimizer.zero_grad()
        out = model(x)
        raw_loss = criterion(out, y) 
        batch_weights = torch.tensor([class_weight_dict[int(yi.item())] for yi in y], dtype=torch.float32)
        loss = (raw_loss * batch_weights.unsqueeze(1)).mean()
        
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_loader.dataset)
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            out = model(x)
            raw_loss = criterion(out, y)
            batch_weights = torch.tensor([class_weight_dict[int(yi.item())] for yi in y], dtype=torch.float32)
            loss = (raw_loss * batch_weights.unsqueeze(1)).mean()
            val_loss += loss.item() * x.size(0)
    val_loss /= len(val_loader.dataset)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

Epoch 1: train_loss=0.0890, val_loss=0.0934
Epoch 2: train_loss=0.0896, val_loss=0.0919
Epoch 3: train_loss=0.0879, val_loss=0.0910
Epoch 4: train_loss=0.0879, val_loss=0.0902
Epoch 5: train_loss=0.0856, val_loss=0.0896
Epoch 6: train_loss=0.0872, val_loss=0.0891
Epoch 7: train_loss=0.0851, val_loss=0.0885
Epoch 8: train_loss=0.0845, val_loss=0.0878
Epoch 9: train_loss=0.0849, val_loss=0.0869
Epoch 10: train_loss=0.0851, val_loss=0.0859
Epoch 11: train_loss=0.0828, val_loss=0.0848
Epoch 12: train_loss=0.0795, val_loss=0.0841
Epoch 13: train_loss=0.0785, val_loss=0.0836
Epoch 14: train_loss=0.0773, val_loss=0.0834
Epoch 15: train_loss=0.0755, val_loss=0.0830
Epoch 16: train_loss=0.0747, val_loss=0.0829
Epoch 17: train_loss=0.0724, val_loss=0.0826
Epoch 18: train_loss=0.0710, val_loss=0.0825
Epoch 19: train_loss=0.0706, val_loss=0.0831
Epoch 20: train_loss=0.0687, val_loss=0.0831
Epoch 21: train_loss=0.0665, val_loss=0.0833
Epoch 22: train_loss=0.0659, val_loss=0.0841
Epoch 23: train_los

## 9. EVALUATION

In [ ]:
model.eval()
y_probs, y_true = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x)
        y_probs.extend(out.numpy())
        y_true.extend(y.numpy())

y_probs = np.array(y_probs)
y_true = np.array(y_true)
y_pred = (y_probs >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred) .ravel()
print("\n=== METRICS ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall: {recall_score(y_true, y_pred):.4f}")
print(f"Specificity: {tn / (tn + fp):.4f}")
print(f"F1-score: {f1_score(y_true, y_pred):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_true, y_probs):.4f}")
print(f"PR AUC: {average_precision_score(y_true, y_probs):.4f}")
print(f"MCC: {matthews_corrcoef(y_true, y_pred):.4f}")
print(f"Cohen Kappa: {cohen_kappa_score(y_true, y_pred):.4f}")
print(f"Brier Score: {brier_score_loss(y_true, y_probs):.4f}")


=== METRICS ===
Accuracy: 0.8198
Precision: 0.8020
Recall: 1.0000
Specificity: 0.3333
F1-score: 0.8901
Balanced Accuracy: 0.6667
ROC AUC: 0.6959
PR AUC: 0.8347
MCC: 0.5170
Cohen Kappa: 0.4219
Brier Score: 0.1536


## 10. KAN eploration

### KAN VISUALIZATION